# Tutorial: MNIST Walkthrough

Audience:
- Reviewers, coauthors, and readers who want one clean MNIST study they can inspect step by step.

Prerequisites:
- Basic PyTorch familiarity.
- A working repo environment with `torch`, `torchvision`, and `matplotlib`.

Learning goals:
- Build the MNIST study config in place.
- Train a small GDFM model directly in notebook cells.
- Sample from the trained model and compute memorization metrics against the training set.

This notebook is intentionally pedagogical and mostly in-memory. The official artifact-writing path lives in `python -m experiments.mnist.run`.


## Outline

1. Pick a few top-level knobs.
2. Import the live repo code and set up the environment.
3. Load a deterministic MNIST subset and inspect it.
4. Build the study config line by line.
5. Instantiate the model and train it in visible cells.
6. Sample from the trained model and measure memorization.
7. Reflect on what changed and try one exercise.


## 0. Knobs

Only touch this cell first. Everything else below reads from these values.


In [ ]:
FAST_NOTEBOOK = True
SEED = 7
N_TRAIN = 8 if FAST_NOTEBOOK else 64
N_EVAL = 16 if FAST_NOTEBOOK else 128
TRAIN_STEPS = 20 if FAST_NOTEBOOK else 200
SAMPLE_STEPS = 8 if FAST_NOTEBOOK else 24
N_GENERATED = 8 if FAST_NOTEBOOK else 24
BATCH_SIZE = min(8, N_TRAIN) if FAST_NOTEBOOK else 16
PREDICTION_SPACE = "v"
LOSS_SPACE = "v"
DEVICE = "auto"
DOWNLOAD = True
DATA_ROOT = "tmp/data"


## 1. Imports and repo path setup

The notebook adds both the repo root and `src/` to `sys.path` so it works whether or not the checkout has already been installed in editable mode.


In [ ]:
from __future__ import annotations

import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision import datasets

repo_root = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / ".git").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise RuntimeError("Could not find the repo root from the current notebook session.")

for candidate in (repo_root, repo_root / "src"):
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)

from gdfm import GDFM
from mp_config import get_base_config
from mp_metrics import compute_memorization_metrics, compute_nn_stats


## 2. Make the run deterministic and choose the device

We seed Python and PyTorch explicitly so the subset choice, batches, and sampling noise are reproducible.


In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

if DEVICE == "auto":
    if torch.cuda.is_available():
        DEVICE = "cuda"
    elif torch.backends.mps.is_available():
        DEVICE = "mps"
    else:
        DEVICE = "cpu"

data_root = repo_root / DATA_ROOT
DEVICE


## 3. Load the raw MNIST splits

We start from the raw torchvision tensors so the preprocessing stays visible in the notebook instead of being hidden behind a loader call.


In [ ]:
train_set = datasets.MNIST(root=str(data_root), train=True, download=DOWNLOAD)
test_set = datasets.MNIST(root=str(data_root), train=False, download=DOWNLOAD)

len(train_set.data), len(test_set.data)


## 4. Take deterministic subsets for training and reference data

The training subset comes from the training split. The reference subset comes from the test split so we can compare generated samples against held-out real data too.


In [ ]:
train_generator = torch.Generator().manual_seed(SEED)
reference_generator = torch.Generator().manual_seed(SEED + 1)

train_indices = torch.randperm(len(train_set.data), generator=train_generator)[:N_TRAIN]
reference_indices = torch.randperm(len(test_set.data), generator=reference_generator)[:N_EVAL]

train_uint8 = train_set.data[train_indices].clone()
reference_uint8 = test_set.data[reference_indices].clone()
train_labels = train_set.targets[train_indices].clone()
reference_labels = test_set.targets[reference_indices].clone()

train_uint8.shape, reference_uint8.shape


## 5. Normalize to `[-1, 1]` and resize to 32x32

The live MNIST study uses a small image-space UNet, so we make the preprocessing explicit here: convert to float, add a channel dimension, scale to `[-1, 1]`, and resize from 28x28 to 32x32.


In [ ]:
train_tensor = train_uint8.float().div(127.5).sub(1.0).unsqueeze(1)
reference_tensor = reference_uint8.float().div(127.5).sub(1.0).unsqueeze(1)

train_tensor = F.interpolate(train_tensor, size=(32, 32), mode="bilinear", align_corners=False)
reference_tensor = F.interpolate(reference_tensor, size=(32, 32), mode="bilinear", align_corners=False)

train_tensor.shape, reference_tensor.shape


## 6. Look at the actual training images

Before we train anything, it is worth seeing exactly what the model will memorize or generalize from.


In [ ]:
n_show = min(6, train_tensor.shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2), squeeze=False)
axes = axes[0]

for index in range(n_show):
    axes[index].imshow(train_tensor[index, 0].cpu(), cmap="gray", vmin=-1.0, vmax=1.0)
    axes[index].set_title(f"label={int(train_labels[index])}")
    axes[index].axis("off")

fig.suptitle(f"Training subset (N={N_TRAIN})")
plt.tight_layout()


## 7. Build the study config in place

This cell uses the same config object shape as the saved-run pipeline, but we fill it in directly so the notebook stays readable and interactive.


In [ ]:
config = get_base_config()

config.study.name = "mnist_walkthrough"
config.study.description = "Pedagogical in-memory MNIST walkthrough."

config.framework.family = "fm"
config.framework.prediction_space = PREDICTION_SPACE
config.framework.loss_space = LOSS_SPACE

config.data.dataset = "mnist"
config.data.root = str(data_root)
config.data.download = DOWNLOAD
config.data.n_train = int(N_TRAIN)
config.data.n_eval = int(N_EVAL)
config.data.image_size = 32
config.data.image_channels = 1
config.data.flatten = False
config.data.data_dim = 32 * 32

config.network.arch = "unet"
config.network.base_channels = 32
config.network.channel_mult = (1, 2, 2)
config.network.attn_resolutions = (8,)
config.network.num_res_blocks = 1
config.network.time_emb_dim = 32
config.network.dropout = 0.0

config.training.seed = int(SEED)
config.training.n_steps = int(TRAIN_STEPS)
config.training.batch_size = int(BATCH_SIZE)
config.training.lr = 3.0e-4
config.training.ckpt_every = max(1, TRAIN_STEPS // 4)
config.training.log_every = max(1, TRAIN_STEPS // 5)

config.evaluation.n_samples = int(N_GENERATED)
config.evaluation.batch_size = int(BATCH_SIZE)
config.evaluation.ratio_threshold = 0.5
config.evaluation.sample_panel_count = min(8, N_GENERATED)
config.evaluation.chunk_size = max(32, N_EVAL, N_GENERATED)

config.sampling.method = "heun"
config.sampling.n_steps = int(SAMPLE_STEPS)

config.framework.prediction_space, config.framework.loss_space, config.training.n_steps


## 8. Instantiate the model and optimizer

At this point we have not hidden anything behind `train_run(...)`. We are building the exact objects we need and then training them directly below.


In [ ]:
model = GDFM(config).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=float(config.training.lr))
n_params = sum(parameter.numel() for parameter in model.parameters())

print(f"device={DEVICE} | parameters={n_params / 1e6:.2f}M | pred={config.framework.prediction_space} | loss={config.framework.loss_space}")


## 9. Train in the notebook

The loop below is intentionally visible. We sample batches from the fixed subset, call `compute_loss(...)`, and update the model in place so the learning flow is easy to inspect.


In [ ]:
loss_history = []
batch_generator = torch.Generator().manual_seed(SEED + 2)

model.train()
for step in range(1, int(config.training.n_steps) + 1):
    batch_indices = torch.randint(0, train_tensor.shape[0], (int(config.training.batch_size),), generator=batch_generator)
    batch = train_tensor[batch_indices].to(DEVICE)

    optimizer.zero_grad(set_to_none=True)
    loss = model.compute_loss(batch)
    loss.backward()
    optimizer.step()

    loss_history.append(float(loss.item()))

    if step == 1 or step % int(config.training.log_every) == 0 or step == int(config.training.n_steps):
        print(f"step {step:03d} | loss={loss.item():.4f}")


## 10. Plot the loss curve

This will not prove anything on its own, but it gives us a quick read on whether the optimization is moving at all.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(loss_history, color="tab:blue")
ax.set_title("Notebook training loss")
ax.set_xlabel("step")
ax.set_ylabel("loss")
plt.tight_layout()


## 11. Sample from the trained model

We now create fresh noise, integrate the learned ODE, and bring the samples back to CPU for inspection.


In [ ]:
sample_generator = torch.Generator().manual_seed(SEED + 3)
z_init = torch.randn((N_GENERATED, 1, 32, 32), generator=sample_generator)

model.eval()
with torch.no_grad():
    generated = model.sample(
        z_init.to(DEVICE),
        n_steps=int(config.sampling.n_steps),
        method=str(config.sampling.method),
    ).cpu()

generated.shape


## 12. Inspect the generated images

Even a tiny walkthrough run can already show whether the model is producing digit-like structure or just noisy textures.


In [ ]:
n_show = min(6, generated.shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2), squeeze=False)
axes = axes[0]

for index in range(n_show):
    axes[index].imshow(generated[index, 0], cmap="gray", vmin=-1.0, vmax=1.0)
    axes[index].axis("off")

fig.suptitle("Generated samples")
plt.tight_layout()


## 13. Compute memorization metrics against the training subset

The core question is how close generated samples are to the training set. We also compare a held-out reference subset against that same training set to keep a baseline in view.


In [ ]:
train_metrics = compute_memorization_metrics(
    generated,
    train_tensor,
    ratio_threshold=float(config.evaluation.ratio_threshold),
)

reference_metrics = compute_memorization_metrics(
    reference_tensor[: generated.shape[0]],
    train_tensor,
    ratio_threshold=float(config.evaluation.ratio_threshold),
)

train_metrics, reference_metrics


## 14. Pull out nearest neighbors for a visual sanity check

The scalar `f_mem` is useful, but it is even more convincing when we can look at each generated image beside its nearest training neighbor.


In [ ]:
neighbor_stats = compute_nn_stats(generated, train_tensor)
nearest_neighbors = train_tensor[neighbor_stats["indices"]]

n_show = min(4, generated.shape[0])
fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 4), squeeze=False)

for index in range(n_show):
    axes[0, index].imshow(generated[index, 0], cmap="gray", vmin=-1.0, vmax=1.0)
    axes[0, index].set_title("generated")
    axes[0, index].axis("off")

    axes[1, index].imshow(nearest_neighbors[index, 0], cmap="gray", vmin=-1.0, vmax=1.0)
    axes[1, index].set_title("nearest train")
    axes[1, index].axis("off")

plt.tight_layout()


## 15. Read the result in one compact summary

This is the moment to compare the generated-sample memorization score to the held-out reference baseline.


In [ ]:
summary = {
    "prediction_space": config.framework.prediction_space,
    "loss_space": config.framework.loss_space,
    "n_train": int(config.data.n_train),
    "train_f_mem": round(train_metrics["f_mem"], 4),
    "reference_f_mem": round(reference_metrics["f_mem"], 4),
    "train_mean_nn_ratio": round(train_metrics["mean_nn_ratio"], 4),
    "reference_mean_nn_ratio": round(reference_metrics["mean_nn_ratio"], 4),
}

summary


## Exercise

Pick exactly one knob in the top cell, rerun the notebook, and compare the summary at the end.

Good first choices:
- switch `PREDICTION_SPACE` from `"v"` to `"x"` or `"eps"`
- make `N_TRAIN` smaller to push the model deeper into the memorization regime
- turn off `FAST_NOTEBOOK` to let the run go longer


In [ ]:
# Fill this in after you rerun the notebook with one changed knob.
exercise_notes = {
    "changed_knob": "PREDICTION_SPACE = 'x'",
    "what_shifted": "",
    "why_i_think_it_happened": "",
}

exercise_notes


## Pitfall and extension

Common mistake:
- comparing generated samples to the test set when you mean to measure memorization against the training set. `f_mem` is defined here relative to the training subset.

Easy extension:
- once you understand the in-memory walkthrough, run `python -m experiments.mnist.run` to create the official saved run with checkpoints and replay metrics.
